# 相対評価（5遺伝子から1つ選ぶ）で創薬標的の可能性を採点する

ノートブック 03 は遺伝子1つずつに Yes/No で答えさせる**絶対評価**です。このノートブックは、5遺伝子を番号付きで並べて「3条件のどれかに最もよく当てはまるのはどれか」を**番号で**選ばせる**相対評価**です。最初の2ラウンドはランダムに5つずつ組み、3ラウンド目からは**それまでの強さが近い遺伝子どうしで組む「スイス式トーナメント」**にします。全ラウンドの勝敗から **Bradley-Terry モデル**（多者択一への拡張＝Luce の選択モデル）で強さを推定して順位を付けます。

- 絶対評価の「Yes と言いやすい／言いにくい」という癖（尤度の偏り）の影響を受けにくい。
- 両方（03 と 04）で上位に来る遺伝子は、聞き方の癖ではなく中身で選ばれている可能性が高い。**03 の結果を確かめる別の物差し**として使ってください。

## 質問 M2r（ノートブック 03 と同じ3つの考え方を「最もよく当てはまるのはどれか」で聞く）

| 条件 | 中身 |
|---|---|
| (a) | 阻害または活性化すると、疾患を治療する、または上に挙げた症状の少なくとも1つを改善する、という仮説を構築できる |
| (b) | 原因経路に含まれていなくても、同じ細胞の並行・拮抗する経路を通じて異常な過程を打ち消す・補える |
| (c) | 自身の基質・リガンド・シグナル経路・細胞・回路が、記載した機構と正確に一致する（同じ遺伝子ファミリーでも似て非なる役割なら不可） |

選択肢は5遺伝子（1〜5番）と「6. None of the above genes seem clearly relevant（該当なし）」です。

## 実機検証の結果（txgemma-9b-chat Q6_K、5疾患 × 100遺伝子、8ラウンド、`scripts/rank_variants.py`）

| 質問（選択式） | AUC 既知 vs ダミー（平均/最低） | AUC 既知 vs その他 | AUC 候補 vs ダミー |
|---|---|---|---|
| 旧 target（あらゆる証拠を総合して最も有望な標的） | 0.904 / 0.807 | 0.804 | 0.872 |
| 旧 evidence（既存薬・ヒト遺伝学の証拠が最も強い） | 0.911 / 0.851 | 0.787 | 0.881 |
| 旧 mechanism（自身の基質・経路が発症の中心） | 0.864 / 0.747 | 0.731 | 0.840 |
| M1r（3条件、(a) が「治療薬が治療する」） | 0.957 / 0.895 | **0.838** | 0.878 |
| **M2r（この notebook。(a) を症状改善まで拡張）** | **0.964 / 0.929** | 0.836 | **0.879** |
| M3sr（M2r を短く言い切った版） | 0.941 / 0.890 | 0.824 | 0.874 |
| （参考）ノートブック 03 の Yes/No M3s | 0.984 / 0.965 | 0.868 | 0.909 |

- 旧版の3パターンは、上の「読む位置」の修正後に測り直した値です。具体的な3条件を並べた M1r / M2r の方がはっきり上でした（「最も有望」「最も証拠が強い」のような広い聞き方は、有名な遺伝子に流れやすい）。
- M1r と M2r の差は誤差の範囲ですが、M2r は最も悪い疾患でも AUC 既知 vs ダミー 0.929 と安定しており、条件もノートブック 03 と同じ考え方なので M2r を採用しました。
- Yes/No で効いた「短く言い切る」書き方（M3sr）は、選択式では効きませんでした。Yes/No では甘い Yes を減らす効果でしたが、選択式は遺伝子どうしの比較なのでその効果が出ないためと考えられます。
- 精度は Yes/No（ノートブック 03）の方が少し上です。**主な採点は 03、このノートブックはその確認用**という位置づけです。前立腺がんのように選択式の方が既知遺伝子を上に置ける疾患もあります（AUC 既知 vs その他 0.880 vs 0.839）。
- 5遺伝子から「外す1つ」を選ぶ、0〜9 の評点、「1つだけ開発するなら」「専門家なら」などの聞き方も試しましたが、いずれも M2r より劣りました（`scripts/ask_modes.py`、`scripts/rank_variants.py`）。

### 組み方：スイス式 × Bradley-Terry（このノートブックの方式）
同じ M2r・同じ8ラウンド（＝同じ質問数）で、グループの組み方と点数の出し方を比べました（`scripts/swiss.py`、`scripts/plot_swiss.py`）。

| 組み方 × 点数 | AUC 既知 vs その他 | 既知 vs ダミー | 候補 vs ダミー | 既知の平均順位 |
|---|---|---|---|---|
| ランダム × mean_p（旧） | 0.836 | 0.964 | **0.879** | 19.0 |
| ランダム × Bradley-Terry | 0.809 | 0.915 | 0.845 | 21.6 |
| **スイス式 × Bradley-Terry（この notebook）** | **0.851** | **0.984** | 0.878 | **17.6** |
| スイス式 × Elo | 0.789 | 0.911 | 0.843 | 23.5 |

- 5疾患中4疾患でランダム × mean_p を上回りました（前立腺がんのみ 0.880 vs 0.863）。シスチン尿症では、正解と同じ SLC ファミリーのダミーとの見分け（AUC）が 0.611 → 0.833 に大きく改善しました。
- ランダムな組み合わせでは相手の大半が弱いダミーなので、Bradley-Terry の「相手の強さで補正する」働きが効きません。スイス式で強さの近い遺伝子どうしを組むと効くようになります。
- Elo は対戦の順番に左右されるため、後半ほど強い相手と当たるスイス式とは相性が悪い（全データをまとめて推定する Bradley-Terry を使う）。


## 旧版からの修正点
- **答えの番号を読む位置を直しました。** 旧版は「Answer:」の直後で数字の確率を読んでいましたが、Gemma には「 1」のような空白付きの数字トークンが無いため、そこでは空白（約85%）が予測され、数字の確率は中身と無関係なノイズでした（「1番と4番ばかり選ばれる」偏りの原因）。今は「**Answer: **」（末尾の空白まで）を入れてから読みます。数字が確率の 90% 以上を占めることを実機で確認済みです。
- **質問は1つに絞り、独立に聞きます。** 旧版は3パターン（target / mechanism / evidence）を、前の答えを引き継ぐ1本の流れで聞いていました。
- **グループの人数は5に固定**しました（「該当なし」を含めて選択肢6つ＝1桁の数字に収まる）。
- **グループの組み方をスイス式に、点数を Bradley-Terry に変えました。** 旧版はランダムな組み合わせ × 選ばれた確率の平均（mean_p）でした（比較は上の表）。
- 質問文・前置きは `scripts/rank_variants.py`、組み方と推定の手順は `scripts/swiss.py` と同じです（変えたら検証し直してください）。

## 前提
- 本命は **`~/llm/models` の GGUF を llama-cpp-python で直接動かす**経路です（前置きの KV キャッシュを保存・復元）。Ollama は前置きを毎回処理するため遅くなります。
- モデルが無い環境ではモック（擬似乱数）で動きます。数値に意味はありません。
- 100遺伝子 × 8ラウンド ＝ 160グループで、GGUF（txgemma-9b Q6_K、Mac）なら約5〜6分です。
- **Bradley-Terry モデル**：遺伝子 i がグループから選ばれる確率を π_i ÷（グループ内の π の合計、「該当なし」を含む）と置き、全ラウンドの結果に最もよく合う強さ π を推定します。「該当なし」も1つの項目として強さを持つので、「該当なしより強いか」も分かります（ただし「該当なし」がほとんど選ばれない疾患では目安になりません）。

### このセルがすること：準備

In [1]:
import os, re, math, glob, json, time, random, hashlib, urllib.request
import numpy as np
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 60)
ROOT = os.path.abspath("..")
print("ready:", ROOT)

ready: /Users/yoshinorisatomi/Documents/claude/gene_disease_prediction02


## 疾患の選択

### このセルがすること：`data/diseases.json` の登録疾患から1つ選び、病名・症状の箇条書き・遺伝子リストを読み込む
- ノートブック 03 と同じ読み込み方です。`DISEASE_KEY` を変えるだけで切り替わります：`ra` / `scz` / `cystinuria` / `prostate_cancer` / `achondroplasia`。

In [2]:
DISEASE_KEY = "ra"                    # "ra" / "scz" / "cystinuria" / "prostate_cancer" / "achondroplasia"
GENE_SET = "set100"                   # "set100" / "set1000" / "known" / "candidates"

REGISTRY = json.load(open(os.path.join(ROOT, "data", "diseases.json"), encoding="utf-8"))
print("登録疾患:", {k: v["name"] for k, v in REGISTRY.items()})
D = REGISTRY[DISEASE_KEY]
DISEASE = D["name"]
DISEASE_INFO = D["info"][:5]
GENE_FILE = os.path.join(ROOT, "data", "genes", f"{D['gene_prefix']}_{GENE_SET}.tsv")
print("選択:", DISEASE, "| 遺伝子リスト:", os.path.basename(GENE_FILE))
for b in DISEASE_INFO: print("  -", b[:110] + ("…" if len(b) > 110 else ""))

登録疾患: {'ra': 'rheumatoid arthritis', 'scz': 'schizophrenia', 'cystinuria': 'cystinuria', 'prostate_cancer': 'prostate cancer', 'achondroplasia': 'achondroplasia'}
選択: rheumatoid arthritis | 遺伝子リスト: ra_set100.tsv
  - Joint pain, swelling and morning stiffness, symmetric in the small joints of hands and feet: caused by chronic…
  - Progressive joint deformity and loss of function: invasive growth of the synovial lining cells and excessive b…
  - The inflammation is sustained by the adaptive immune system: self-reactive T cells, antibody-producing B cells…
  - Fatigue, low-grade fever and anaemia: systemic effects of inflammatory mediators released by the activated imm…
  - Accelerated cardiovascular disease and interstitial lung disease: consequences of long-standing systemic infla…


## 設定

### このセルがすること：ラウンド数とモデルの選択を宣言する
- `N_ROUNDS`：ラウンド数（＝1遺伝子あたりの比較回数、検証は 8）。`RANDOM_ROUNDS`：最初にランダムに組むラウンド数（検証は 2）。
- `JITTER`：スイス式で組むとき、強さに足す乱数の大きさ（強さの標準偏差に対する比。毎回まったく同じ組にならないようにする。検証は 0.3）。
- `SEED`：乱数。検証スクリプトと同じ 0 にすると、同じ組み方になります。
- モデルの選び方はノートブック 03 と同じ仕組み（`~/llm/models` の GGUF、または Ollama）です。

In [3]:
MAX_GENES = None                                   # 試運転なら 20 など
N_ROUNDS = 8                                       # ≒ 1遺伝子あたりの比較回数
RANDOM_ROUNDS = 2                                  # 最初にランダムに組むラウンド数。残りはスイス式
JITTER = 0.3
SEED = 0
GROUP_SIZE = 5                                     # 固定（「該当なし」を含めて選択肢6つ＝1桁の数字）
N_CTX = 2048

# --- モデルの選択 ---
BACKEND = "auto"                                   # "auto" / "gguf" / "ollama" / "mock"
MODEL_DIR = os.path.expanduser("~/llm/models")
OLLAMA_URL = "http://localhost:11434"
MODEL_SELECT = "auto"                              # "auto" / 一覧の番号 / 名前の一部
PREFER = ("txgemma", "medgemma", "gemma")
models = []
for h in sorted(glob.glob(os.path.join(MODEL_DIR, "**", "*.gguf"), recursive=True)):
    models.append({"kind": "gguf", "name": os.path.basename(h), "path": h, "size_gb": round(os.path.getsize(h) / 1e9, 2)})
try:
    with urllib.request.urlopen(OLLAMA_URL + "/api/tags", timeout=3) as r:
        for m in json.load(r).get("models", []):
            models.append({"kind": "ollama", "name": m["name"], "path": m["name"], "size_gb": round(m.get("size", 0) / 1e9, 2)})
except Exception:
    pass
chosen = None
if BACKEND != "mock":
    if isinstance(MODEL_SELECT, int):
        chosen = models[MODEL_SELECT]
    else:
        pool = [m for m in models if BACKEND == "auto" or m["kind"] == BACKEND]
        if MODEL_SELECT != "auto": pool = [m for m in pool if MODEL_SELECT.lower() in m["name"].lower()]
        ranked = sorted(pool, key=lambda m: (min([i for i, p in enumerate(PREFER) if p in m["name"].lower()] or [99]),
                                             0 if m["kind"] == "gguf" else 1, m["name"]))
        chosen = ranked[0] if ranked else None
USE_LLM = chosen is not None
BACKEND_USED = chosen["kind"] if chosen else "mock"
MODEL_PATH = chosen["path"] if chosen else None
print("使えるモデル:"); [print(f"  [{i}] {m['kind']:6s} {m['size_gb']:6.2f} GB  {m['name']}") for i, m in enumerate(models)]
print("選択:", f"{BACKEND_USED}: {MODEL_PATH}" if USE_LLM else "モック（擬似乱数。数値に意味なし）")
OUT_DIR = os.path.join(ROOT, "outputs"); os.makedirs(OUT_DIR, exist_ok=True)

使えるモデル:
  [0] gguf     7.59 GB  txgemma-9b-chat-Q6_K.gguf
  [1] ollama   7.59 GB  txgemma-9b-chat-q6_k:latest
  [2] ollama  17.40 GB  gemma3:27b
  [3] ollama   4.68 GB  qwen2.5:7b
  [4] ollama   1.16 GB  bge-m3:latest
  [5] ollama   9.28 GB  qwen3:14b
  [6] ollama   4.37 GB  cniongolo/biomistral:latest
  [7] ollama   4.01 GB  gemma3:4b-it-qat
  [8] ollama   5.23 GB  deepseek-r1:8b
  [9] ollama   5.23 GB  qwen3:8b
  [10] ollama   8.99 GB  qwen2.5:14b
  [11] ollama   4.92 GB  llama3.1:latest
選択: gguf: /Users/yoshinorisatomi/llm/models/txgemma-9b-chat-Q6_K.gguf


## 入力遺伝子

### このセルがすること：遺伝子リストを読み、プロンプトに入れる名前（記号＋タンパク質名）を作る
- ノートブック 03 と同じ形式です。

In [4]:
genes = pd.read_csv(GENE_FILE, sep="\t", dtype=str).fillna("")
for col in ("protein_name_uniprot", "gene_name", "category", "label"):
    if col not in genes.columns: genes[col] = ""
if MAX_GENES: genes = genes.head(MAX_GENES).copy()
genes["gene_label"] = [f"{g['symbol']} ({(g['protein_name_uniprot'] or g['gene_name']).split('|')[0]})" if (g["protein_name_uniprot"] or g["gene_name"]) else g["symbol"] for _, g in genes.iterrows()]
print(len(genes), "genes"); genes[["symbol", "gene_label", "category"]].head(5)

100 genes


,symbol,gene_label,category
0,PADI4,PADI4 (peptidyl arginine deiminase 4),candidate
1,CCL21,CCL21 (C-C motif chemokine ligand 21),candidate
2,TNFRSF14,TNFRSF14 (TNF receptor superfamily member 14),candidate
3,IL7R,IL7R (interleukin 7 receptor),candidate
4,IRF5,IRF5 (interferon regulatory factor 5),candidate


## 質問 M2r と、キャッシュが効くプロンプトの並び

### このセルがすること：質問を定義し、前置き・番号付き遺伝子リスト・質問文の組み立て方を定義する
- 前置き（`prefix_text`）は役割・注意書き・病名・症状・答え方の指示で、**全グループで1文字も変わらない**ので1回だけ処理して保存します。
- `group_block`：5遺伝子を「1. 記号 (タンパク質名)」の番号付きリストにし、最後に「6. 該当なし」を足します。
- `question_line`：質問文の末尾は「**Answer: **」（空白まで）。この直後で数字1〜6の確率を読みます。

In [5]:
QID = 'M2r'
QUESTION = "Consider these three criteria:\n(a) A hypothesis can be constructed that inhibiting or activating the gene would treat {disease} or improve at least one of the symptoms listed above.\n(b) Even if the gene is not part of the pathway that causes {disease}, activating or inhibiting it could counteract or compensate for the abnormal process described above, for example through a parallel or opposing pathway in the same cells.\n(c) The gene's own specific role — its substrate, ligand, signalling pathway, cell type or circuit — matches the mechanism described above precisely, rather than a related but distinct one (e.g. a different molecule, cell type, tissue or subcellular compartment); a similar-sounding but distinct role, even in the same gene family, does not count.\nWhich numbered gene above best meets at least one of these criteria?"
STRICT_NOTE = 'Note: the vast majority of human genes are NOT drug targets for any given disease. Pick a gene only when there is a clear, specific reason; membership in the same gene family as a known disease gene is NOT sufficient by itself.\n'
OUT_CSV = os.path.join(OUT_DIR, os.path.basename(GENE_FILE).replace(".tsv", "") + f"_rank_{QID.lower()}.csv")   # 例 ra_set100_rank_m3sr.csv

def prefix_text():
    """全グループで共通の前置き（1回だけ処理して保存する）。"""
    bullets = "\n".join(f"- {b}" for b in DISEASE_INFO)
    return ("You are an expert in drug discovery and human disease biology. You will be shown a numbered list of "
            f"{GROUP_SIZE} candidate genes for one disease, and asked which ONE number is the best answer to a question.\n" + STRICT_NOTE +
            f"Disease: {DISEASE}\nTarget symptoms and the organ, cell and functional abnormalities behind them:\n{bullets}\n"
            f"Answer with a single number from 1 to {GROUP_SIZE + 1} only. No words, no explanation.\n\n")

def group_block(labels):
    lines = [f"{i + 1}. {g}" for i, g in enumerate(labels)] + [f"{len(labels) + 1}. None of the above genes seem clearly relevant"]
    return "Candidate genes:\n" + "\n".join(lines) + "\n"

def question_line():
    return f"Question: {QUESTION.format(disease=DISEASE)} Answer: "     # 末尾の空白が必要（数字はこの後に来る）

print(prefix_text() + group_block(["TNF (tumor necrosis factor)", "PADI4 (peptidyl arginine deiminase 4)", "IL6 (interleukin 6)",
                                   "BRCA1 (breast cancer 1)", "ACTB (actin beta)"]) + question_line())

You are an expert in drug discovery and human disease biology. You will be shown a numbered list of 5 candidate genes for one disease, and asked which ONE number is the best answer to a question.
Note: the vast majority of human genes are NOT drug targets for any given disease. Pick a gene only when there is a clear, specific reason; membership in the same gene family as a known disease gene is NOT sufficient by itself.
Disease: rheumatoid arthritis
Target symptoms and the organ, cell and functional abnormalities behind them:
- Joint pain, swelling and morning stiffness, symmetric in the small joints of hands and feet: caused by chronic inflammation of the synovial membrane that lines the joints
- Progressive joint deformity and loss of function: invasive growth of the synovial lining cells and excessive bone resorption by bone-degrading cells erode cartilage and bone
- The inflammation is sustained by the adaptive immune system: self-reactive T cells, antibody-producing B cells and in

## エンジン（GGUF を llama-cpp-python で直接動かす）

### `score_group(labels)` — 1グループ（5遺伝子）に質問して、選択肢1〜6の確率を返す
各ステップ：
1. コンテキストを空にしてから、保存しておいた前置きの状態を復元する（前置きは再処理しない）。
2. 番号付き遺伝子リストと質問文を処理する。
3. 「Answer: 」の直後の位置で、数字 1〜6 のトークンの確率を読み、この6つの中で合計1になるよう正規化する。
return：numpy 配列（長さ6。最後が「該当なし」）。

### このセルがすること：エンジンを読み込み、前置きを処理して状態を保存し、上の def を定義する
- 数字 1〜6 がそれぞれ1トークンであることを確認します（2トークンに割れるモデルでは使えません）。
- Ollama では上位20トークンの確率から数字を拾います。logprobs 非対応なら、温度ありで8回生成した数字の割合で代用します。

In [6]:
if BACKEND_USED == "gguf":
    from llama_cpp import Llama
    llm = Llama(model_path=MODEL_PATH, n_ctx=N_CTX, n_gpu_layers=-1, logits_all=False, verbose=False)
    def tok(text, bos=False): return llm.tokenize(text.encode("utf-8"), add_bos=bos, special=bos)
    DIGIT = {n: tok(str(n)) for n in range(1, GROUP_SIZE + 2)}
    assert all(len(v) == 1 for v in DIGIT.values()), f"数字が1トークンになっていません: {DIGIT}"
    llm.reset(); llm.eval(tok(prefix_text(), bos=True))
    PREFIX_STATE = llm.save_state()
    print(f"GGUF engine ready | prefix tokens: {llm.n_tokens} | digit ids {[v[0] for v in DIGIT.values()]}")
elif BACKEND_USED == "ollama":
    print("Ollama を使います（前置きのキャッシュ保存はできないため遅い経路です）:", MODEL_PATH)
else:
    print("警告: モデルが無いのでモック（擬似乱数）です。")

def softmax(l):
    l = np.asarray(l, float); p = np.exp(l - l.max()); return p / p.sum()

def score_group_gguf(labels):
    llm.reset()                                                     # 1a. 前のグループの続きを消す
    llm.load_state(PREFIX_STATE)                                    # 1b. 前置きの状態を復元
    llm.eval(tok(group_block(labels)))                              # 2a. 番号付き遺伝子リスト
    llm.eval(tok(question_line()))                                  # 2b. 質問 → 「Answer: 」の直後（検証スクリプトと同じく別々にトークン化）
    lg = np.ctypeslib.as_array(llm._ctx.get_logits(), shape=(llm.n_vocab(),)).astype(np.float64)
    return softmax([lg[DIGIT[n][0]] for n in range(1, GROUP_SIZE + 2)])   # 3. 数字1〜6だけで正規化

# ---- Ollama（遅い経路）: raw テンプレートで1回ずつ ----
def ollama_template(prompt_body):
    n = str(MODEL_PATH).lower()
    if "gemma" in n:        tpl = "<start_of_turn>user\n{body}<end_of_turn>\n<start_of_turn>model\nAnswer: "
    elif "qwen3" in n:      tpl = "<|im_start|>user\n{body}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nAnswer: "
    elif "qwen" in n or "deepseek" in n: tpl = "<|im_start|>user\n{body}<|im_end|>\n<|im_start|>assistant\nAnswer: "
    elif "mistral" in n:    tpl = "[INST] {body} [/INST] Answer: "
    elif "llama" in n:      tpl = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{body}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nAnswer: "
    else:                   tpl = "{body}\nAnswer: "
    return tpl.format(body=prompt_body)

OLLAMA_LP_ENDPOINT = None

def ollama_post(path, body):
    req = urllib.request.Request(OLLAMA_URL + path, data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.load(r)

def ollama_top_logprobs(full_prompt, k=20):
    """先頭1トークンの上位 k 個の {token: log確率}。両エンドポイントとも非対応なら None。"""
    global OLLAMA_LP_ENDPOINT
    tries = [OLLAMA_LP_ENDPOINT] if OLLAMA_LP_ENDPOINT else ["/api/generate", "/v1/completions"]
    for ep in tries:
        try:
            if ep == "/api/generate":
                out = ollama_post(ep, {"model": MODEL_PATH, "prompt": full_prompt, "raw": True, "stream": False,
                                       "logprobs": True, "top_logprobs": k, "options": {"temperature": 0, "num_predict": 1}})
                lps = out.get("logprobs") or []
                top = {t["token"]: t["logprob"] for t in (lps[0].get("top_logprobs", []) if lps else [])}
                if lps and not top: top = {lps[0]["token"]: lps[0]["logprob"]}
            else:   # /v1/completions では logprobs は「個数」を渡す
                ch = ollama_post(ep, {"model": MODEL_PATH, "prompt": full_prompt, "max_tokens": 1, "temperature": 0,
                                      "logprobs": k})["choices"][0]
                lp = ch.get("logprobs") or {}
                if lp.get("content"): top = {t["token"]: t["logprob"] for t in lp["content"][0].get("top_logprobs", [])}
                elif lp.get("top_logprobs"): top = dict(lp["top_logprobs"][0])
                else: top = {}
            if top:
                OLLAMA_LP_ENDPOINT = ep
                return top
        except Exception:
            continue
    return None

def score_group_ollama(labels):
    body = prefix_text() + group_block(labels) + question_line().replace(" Answer: ", "")
    n_opt = GROUP_SIZE + 1
    top = ollama_top_logprobs(ollama_template(body), 20)
    if top is None:                                                 # logprobs 非対応 → 8回生成した数字の割合で代用
        counts = np.zeros(n_opt)
        for _ in range(8):
            out = ollama_post("/api/generate", {"model": MODEL_PATH, "prompt": ollama_template(body), "raw": True,
                                                "stream": False, "options": {"temperature": 1.0, "num_predict": 3}})
            m = re.match(r"\s*(\d)", out.get("response") or "")
            if m and 1 <= int(m.group(1)) <= n_opt: counts[int(m.group(1)) - 1] += 1
        return (counts + 0.5) / (counts.sum() + 0.5 * n_opt)
    floor = min(top.values())
    def lse(v): m = max(v); return m + math.log(sum(math.exp(x - m) for x in v))
    return softmax([lse([v for t, v in top.items() if t.strip() == str(n)] or [floor]) for n in range(1, n_opt + 1)])

def score_group_mock(labels):
    return softmax([int(hashlib.md5((g + QID).encode()).hexdigest(), 16) % 1000 / 250 for g in labels + ["NONE"]])

score_group = {"gguf": score_group_gguf, "ollama": score_group_ollama}.get(BACKEND_USED, score_group_mock)
demo = ["TNF (tumor necrosis factor)", "ACTB (actin beta)", "IL6 (interleukin 6)", "GAPDH (glyceraldehyde-3-phosphate dehydrogenase)", "ALB (albumin)"]
t0 = time.time(); p = score_group(demo)
print("test:", {i + 1: round(float(v), 3) for i, v in enumerate(p)}, f"| {time.time() - t0:.2f}s")

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


GGUF engine ready | prefix tokens: 267 | digit ids [235274, 235284, 235304, 235310, 235308, 235318]


test: {1: 0.607, 2: 0.0, 3: 0.388, 4: 0.0, 5: 0.0, 6: 0.003} | 1.85s


## 実行（スイス式トーナメント）

### `random_groups(idx, rng)` — ランダムに5つずつ組む（最初の `RANDOM_ROUNDS` ラウンド）
### `swiss_groups(idx, strength, rng, jitter)` — 強さの近い遺伝子どうしで5つずつ組む
どんな def か：強さに少し乱数を足してから強い順に並べ、先頭から5つずつ組みます。組の中の並び（＝番号）はシャッフルして、番号の癖が強さと結び付かないようにします。
### `luce(groups, n)` — Bradley-Terry（Luce の選択モデル）の強さを MM 法で推定する
どんな def か：groups は（メンバーの番号、各メンバーが選ばれた確率）の並び。全勝・全敗でも発散しないよう、各項目に「強さ1の仮想相手との 0.5 勝 0.5 敗」を足します。return：log π。

### このセルがすること：ラウンドをくり返して各グループに質問し、全ラウンドの結果から強さを推定して CSV に書く
- 3ラウンド目からは、それまでの全データで推定した強さを使って組みます。
- 遺伝子ごとの点数：
  - `bt`：Bradley-Terry の強さ（log π）。**順位にはこれを使います。**
  - `above_none`：「該当なし」より強いか。
  - `mean_p`・`win_rate`：選ばれた確率の平均・1位だった割合（参考。スイス式では強い相手と当たるほど下がるので、順位には使わない）。

In [7]:
def random_groups(idx, rng):
    """rank_variants.py と同じ手順（シャッフル → 先頭から5つずつ → 端数は他から補充）。"""
    order = idx[:]; rng.shuffle(order); out = []
    for start in range(0, len(order), GROUP_SIZE):
        chunk = order[start:start + GROUP_SIZE]
        if len(chunk) < GROUP_SIZE: chunk = chunk + rng.sample([i for i in idx if i not in chunk], GROUP_SIZE - len(chunk))
        out.append(chunk)
    return out


def swiss_groups(idx, strength, rng, jitter):
    """強さの順に並べて先頭から5つずつ組む。強さに乱数（標準偏差 jitter × 強さの標準偏差）を足して毎回少しずらす。
    グループ内の並び（＝番号）はシャッフルして、番号の癖が強さと結び付かないようにする。"""
    s = np.array([strength[i] for i in idx]); s = s + np.array([rng.gauss(0, 1) for _ in idx]) * jitter * (s.std() or 1)
    order = [idx[k] for k in np.argsort(-s)]; out = []
    for start in range(0, len(order), GROUP_SIZE):
        chunk = order[start:start + GROUP_SIZE]
        if len(chunk) < GROUP_SIZE: chunk = chunk + order[start - (GROUP_SIZE - len(chunk)):start]      # 端数は直前（強さの近い）遺伝子で補充
        chunk = chunk[:]; rng.shuffle(chunk); out.append(chunk)
    return out


def luce(groups, n, iters=1000, prior=0.5):
    """Luce の選択モデル（Bradley-Terry を多者択一に広げたもの）の強さを MM 法（Hunter 2004）で推定する。
    groups: [(メンバーの番号の並び, 各メンバーが選ばれた確率)]。遺伝子 i が選ばれる確率 = π_i / Σ_{グループ内} π。
    各項目に「強さ1の仮想相手との prior 勝ち・prior 負け」を足して、全勝・全敗でも発散しないようにする。return: log π"""
    W, pi = np.full(n, prior), np.ones(n)
    for mem, p in groups:
        for m, q in zip(mem, p): W[m] += q
    for _ in range(iters):
        den = 2 * prior / (pi + 1)
        for mem, p in groups:
            den[mem] += 1 / pi[mem].sum()
        new = W / den; new /= np.exp(np.mean(np.log(new)))
        if np.max(np.abs(np.log(new) - np.log(pi))) < 1e-10: pi = new; break
        pi = new
    return np.log(pi)


def fit_luce(rows):
    """これまでの結果（rows）から、遺伝子（genes の行番号の順）と「該当なし」の強さ log π を推定する。"""
    L = pd.DataFrame(rows); items = idx + ["NONE"]; ix = {s: k for k, s in enumerate(items)}
    groups = []
    for _, d in L.groupby(["round", "group"]):
        d = d.sort_values("position")
        groups.append(([ix[i] for i in d["gi"]] + [ix["NONE"]], np.r_[d["p"].values, d["p_none"].iloc[0]]))
    return luce(groups, len(items))

rng, idx, rows, t0 = random.Random(SEED), list(genes.index), [], time.time()
for r in range(N_ROUNDS):
    if r < RANDOM_ROUNDS:
        groups = random_groups(idx, rng)
    else:
        s = fit_luce(rows); groups = swiss_groups(idx, {i: s[k] for k, i in enumerate(idx)}, rng, JITTER)
    for gi, chunk in enumerate(groups):
        p = score_group([genes.at[i, "gene_label"] for i in chunk])
        for slot, i in enumerate(chunk):     # 確率は小数6桁に丸める（検証スクリプトと同じ。丸め方が違うと強さの推定がわずかに変わり、組み方がずれる）
            rows.append({"round": r, "group": gi, "position": slot + 1, "gi": i, "p": round(float(p[slot]), 6), "p_none": round(float(p[-1]), 6),
                         "win": int(np.argmax(p) == slot)})
    print(f"round {r + 1}/{N_ROUNDS} ({'random' if r < RANDOM_ROUNDS else 'swiss'}, {time.time() - t0:.0f}s)")
long = pd.DataFrame(rows)
s = fit_luce(rows); NONE_STRENGTH = float(s[-1])
long["lr"] = np.log(long["p"].clip(1e-9)) - np.log(long["p_none"].clip(1e-9))
agg = long.groupby("gi").agg(mean_p=("p", "mean"), win_rate=("win", "mean"), lr_none=("lr", "mean"), n_trials=("p", "size"))
res = genes[["symbol", "gene_label", "category", "label"]].join(agg)
res["bt"] = s[:-1]; res["above_none"] = res["bt"] > NONE_STRENGTH
res["rank"] = res["bt"].rank(ascending=False, method="min").astype(int)
res["model"] = f"{BACKEND_USED}:{os.path.basename(str(MODEL_PATH))}" if USE_LLM else "MOCK"; res["disease"] = DISEASE; res["question"] = QID
res = res.sort_values("rank").reset_index(drop=True)
res.round(4).to_csv(OUT_CSV, index=False)
long["strength"] = long["gi"].map(dict(zip(idx, s[:-1])))
spread = long.groupby(["round", "group"])["strength"].std().groupby("round").mean()
print(f"elapsed {time.time() - t0:.0f}s | {len(long) // GROUP_SIZE} groups | wrote {OUT_CSV}")
print("同じ組の中の強さのばらつき（ラウンドごと。スイス式に入ると下がる）:", {int(k) + 1: round(v, 2) for k, v in spread.items()})
print(f"「該当なし」の強さ {NONE_STRENGTH:.2f} | 該当なしより強い遺伝子: 既知 {res.loc[res.category == 'known', 'above_none'].mean():.2f}, "
      f"候補 {res.loc[res.category == 'candidate', 'above_none'].mean():.2f}, ダミー {res.loc[res.category == 'random', 'above_none'].mean():.2f}")

round 1/8 (random, 37s)


round 2/8 (random, 80s)


round 3/8 (swiss, 118s)


round 4/8 (swiss, 157s)


round 5/8 (swiss, 194s)


round 6/8 (swiss, 233s)


round 7/8 (swiss, 275s)


round 8/8 (swiss, 316s)
elapsed 316s | 160 groups | wrote /Users/yoshinorisatomi/Documents/claude/gene_disease_prediction02/outputs/ra_set100_rank_m2r.csv
同じ組の中の強さのばらつき（ラウンドごと。スイス式に入ると下がる）: {1: 1.83, 2: 1.75, 3: 1.11, 4: 1.07, 5: 0.89, 6: 0.67, 7: 0.63, 8: 0.56}
「該当なし」の強さ -2.12 | 該当なしより強い遺伝子: 既知 1.00, 候補 0.89, ダミー 0.21


## 評価

### このセルがすること：AUC、番号の癖、「該当なし」の使われ方、既知遺伝子の順位を出す
- `AUC known vs others`：既知を候補・ダミーより上に置けるか。`known vs random`：既知をダミーより上に。`candidate vs random`：候補をダミーより上に。
- **番号の癖**：遺伝子はシャッフルしているので、番号ごとの平均確率は均等（1/6 ≈ 0.17）に近いはずです。大きく偏っていれば、中身ではなく番号に釣られています。
- **ファミリーの後光効果**：ダミーを「正解と同じ遺伝子ファミリー」と「違うファミリー」に分けて known との AUC を比べます。

In [8]:
def auc(pos, neg):
    pos, neg = np.asarray(list(pos), float), np.asarray(list(neg), float)
    if len(pos) == 0 or len(neg) == 0: return float("nan")
    return float(((pos[:, None] > neg[None, :]).sum() + 0.5 * (pos[:, None] == neg[None, :]).sum()) / (len(pos) * len(neg)))
if not USE_LLM: print("警告: モックの数値です。")
known, other, rnd, cand = (res["category"] == "known"), (res["category"] != "known"), (res["category"] == "random"), (res["category"] == "candidate")
display(pd.DataFrame({c: {"AUC known vs others": auc(res.loc[known, c], res.loc[other, c]), "AUC known vs random": auc(res.loc[known, c], res.loc[rnd, c]),
                          "AUC candidate vs random": auc(res.loc[cand, c], res.loc[rnd, c])} for c in ("bt", "mean_p", "win_rate")}).round(3))
print("※ スイス式では mean_p・win_rate は相手の強さに左右されるので、順位には bt を使います。")

pos = long.groupby("position")["p"].mean()
print("番号ごとの平均確率（均等なら 0.167）:", {int(k): round(v, 3) for k, v in pos.items()},
      "| 「該当なし」:", round(long.drop_duplicates(["round", "group"])["p_none"].mean(), 3))

_fam = lambda sym: (re.match(r"^([A-Za-z]+\d+)", sym).group(1) if re.match(r"^([A-Za-z]+\d+)", sym) else sym)
res["family"] = res["symbol"].apply(_fam)
res["same_family_as_known"] = res["family"].isin(set(res.loc[known, "family"]))
rnd_same, rnd_diff = rnd & res["same_family_as_known"], rnd & ~res["same_family_as_known"]
if rnd_same.sum() >= 3 and rnd_diff.sum() >= 3:
    print(f"AUC known vs random（同じファミリー） {auc(res.loc[known, 'bt'], res.loc[rnd_same, 'bt']):.3f} | "
          f"（違うファミリー） {auc(res.loc[known, 'bt'], res.loc[rnd_diff, 'bt']):.3f}")
else:
    print(f"正解と同じファミリーのダミー {int(rnd_same.sum())} 件（3件未満なのでファミリー比較は省略）")

print("\n既知遺伝子の順位:")
display(res.loc[known, ["rank", "symbol", "gene_label", "bt", "above_none", "mean_p"]])
print("上位 15:")
res.head(15)[["rank", "symbol", "category", "bt", "above_none", "mean_p"]]

,bt,mean_p,win_rate
AUC known vs others,0.694,0.689,0.669
AUC known vs random,0.967,0.971,0.905
AUC candidate vs random,0.925,0.949,0.878


※ スイス式では mean_p・win_rate は相手の強さに左右されるので、順位には bt を使います。
番号ごとの平均確率（均等なら 0.167）: {1: 0.32, 2: 0.178, 3: 0.205, 4: 0.148, 5: 0.119} | 「該当なし」: 0.031
正解と同じファミリーのダミー 0 件（3件未満なのでファミリー比較は省略）

既知遺伝子の順位:


,rank,symbol,gene_label,bt,above_none,mean_p
0,1,IL1R1,IL1R1 (interleukin 1 receptor type 1),4.706218,True,0.614682
2,3,IL6,IL6 (interleukin 6),4.238097,True,0.464853
3,4,TNF,TNF (tumor necrosis factor),4.159440,True,0.624608
4,5,JAK2,JAK2 (Janus kinase 2),3.271822,True,0.379690
7,8,JAK1,JAK1 (Janus kinase 1),2.762651,True,0.263536
10,11,IL6R,IL6R (interleukin 6 receptor),2.366096,True,0.348128
19,20,TNFSF11,TNFSF11 (TNF superfamily member 11),1.357785,True,0.316624
28,29,JAK3,JAK3 (Janus kinase 3),1.112453,True,0.265658
34,35,PTGS2,PTGS2 (prostaglandin-endoperoxide synthase 2),0.741407,True,0.257286
48,49,CD86,CD86 (CD86 molecule),0.004024,True,0.267637


上位 15:


,rank,symbol,category,bt,above_none,mean_p
0,1,IL1R1,known,4.706218,True,0.614682
1,2,IL1B,candidate,4.620558,True,0.623295
2,3,IL6,known,4.238097,True,0.464853
3,4,TNF,known,4.159440,True,0.624608
4,5,JAK2,known,3.271822,True,0.379690
5,6,NFKB1,candidate,3.035662,True,0.371067
6,7,IL17RA,candidate,2.984876,True,0.360262
7,8,JAK1,known,2.762651,True,0.263536
8,9,TNFRSF1A,candidate,2.691532,True,0.469799
9,10,CD40LG,candidate,2.464359,True,0.369881


## 対話型グラフ（ホバーで遺伝子名）

### このセルがすること：分類ごとの分布、順位付きの全遺伝子図、番号ごとの平均確率を描く
- 各点にマウスを乗せると **記号・タンパク質名・分類・順位・値** が出ます。凡例をクリックすると表示／非表示を切り替えられます。
- 同じ図を `outputs/<セット名>_rank_<質問ID>_charts.html` にも保存します。plotly.js を HTML に埋め込むので、ネット接続なしで開けます。
- `pip install plotly nbformat ipython` が必要です（入れた後はカーネルを再起動）。

In [9]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

CAT = [("known", "#2a78d6", "circle", "known"), ("candidate", "#eb6834", "square", "candidate"),
       ("random_other", "#1baf7a", "diamond", "random"), ("random_same", "#eda100", "triangle-up", "random（正解と同じファミリー）")]
MASK = {"known": known, "candidate": cand, "random_other": rnd & ~res["same_family_as_known"], "random_same": rnd & res["same_family_as_known"]}
CAT = [c for c in CAT if MASK[c[0]].any()]
hover = "<b>%{customdata[0]}</b><br>%{customdata[1]}<br>%{customdata[2]}<br>順位 %{customdata[3]}<br>%{y:.3f}<extra></extra>"
cd = lambda d: d[["symbol", "gene_label", "category", "rank"]].values
rng_j = random.Random(0)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Bradley-Terry の強さ bt（順位に使う。点線＝「該当なし」）", "mean_p（参考）"))
for c, col in enumerate(("bt", "mean_p"), start=1):
    for j, (cat, color, sym, name) in enumerate(CAT):
        d = res[MASK[cat]]
        fig.add_trace(go.Scatter(x=[j + (rng_j.random() - 0.5) * 0.5 for _ in range(len(d))], y=d[col], mode="markers",
                                 name=name, legendgroup=cat, showlegend=(c == 1),
                                 marker=dict(color=color, symbol=sym, size=9 if cat != "candidate" else 7, opacity=0.8, line=dict(width=1, color="#fcfcfb")),
                                 customdata=cd(d), hovertemplate=hover), row=1, col=c)
    fig.update_xaxes(tickvals=list(range(len(CAT))), ticktext=[x[3].split("（")[0] for x in CAT], row=1, col=c)
fig.add_hline(y=NONE_STRENGTH, line=dict(color="#0b0b0b", dash="dot"), row=1, col=1)
fig.update_layout(height=440, width=1100, title=f"{DISEASE}: {QID} (pick 1 of 5, Swiss rounds) by category (hover = gene)", template="plotly_white")
fig.show()

fig2 = go.Figure()
for cat, color, sym, name in CAT:
    d = res[MASK[cat]]
    fig2.add_trace(go.Scatter(x=d["rank"], y=d["bt"], mode="markers", name=name,
                              marker=dict(color=color, symbol=sym, size=9, line=dict(width=1, color="#fcfcfb")), customdata=cd(d), hovertemplate=hover))
kn = res[known]
fig2.add_trace(go.Scatter(x=kn["rank"], y=kn["bt"], mode="text", text=kn["symbol"], textposition="top center",
                          textfont=dict(size=10, color="#2a78d6"), showlegend=False, hoverinfo="skip"))
fig2.update_layout(title=f"{DISEASE}: all genes ranked by Bradley-Terry strength (known genes labelled)", xaxis_title="rank", yaxis_title="bt (log π)",
                   template="plotly_white", width=1100, height=440)
fig2.show()

pos_all = list(pos.values) + [long.drop_duplicates(["round", "group"])["p_none"].mean()]
fig3 = go.Figure(go.Bar(x=[f"{i}番" for i in pos.index] + ["該当なし"], y=pos_all, marker_color=["#2a78d6"] * len(pos) + ["#c9c8c2"],
                        hovertemplate="%{x}<br>平均確率 %{y:.3f}<extra></extra>"))
fig3.add_hline(y=1 / (GROUP_SIZE + 1), line=dict(color="#0b0b0b", dash="dot"), annotation_text="均等なら 1/6")
fig3.update_layout(title="番号ごとの平均確率（遺伝子はシャッフルしているので、偏りは番号の癖）", template="plotly_white", width=700, height=380)
fig3.show()

html_path = OUT_CSV.replace(".csv", "_charts.html")
with open(html_path, "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'></head><body>")
    for i, fg in enumerate((fig, fig2, fig3)):
        f.write(fg.to_html(full_html=False, include_plotlyjs=True if i == 0 else False))
    f.write("</body></html>")
print("saved:", html_path)

saved: /Users/yoshinorisatomi/Documents/claude/gene_disease_prediction02/outputs/ra_set100_rank_m2r_charts.html
